# IMPORTS

In [1]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
import time
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import warnings

### Suppress pandas fragmentation warnings

In [2]:
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

---

# 1. SETUP PATHS & MEDIAPIPE

In [3]:
CSV_PATH = r"./dataset/mediapipe_dataset.csv"
# --- NEW: Path for the isolated new data ---
NEW_CSV_PATH = r"./dataset/newly_added_signs.csv" 
MODEL_PATH = 'hand_landmarker.task'

print("Loading MediaPipe...")
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5
)
detector = vision.HandLandmarker.create_from_options(options)

Loading MediaPipe...


---

# 2. DRAWING FUNCTION

### HAND_CONNECTIONS

In [4]:
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         
    (0, 5), (5, 6), (6, 7), (7, 8),         
    (9, 10), (10, 11), (11, 12),            
    (13, 14), (14, 15), (15, 16),           
    (17, 18), (18, 19), (19, 20),           
    (0, 17), (5, 9), (9, 13), (13, 17)      
]

### DRAW_CUSTOM_LANDMARKS

In [5]:
def draw_custom_landmarks(image, landmarks):
    h, w, c = image.shape
    for connection in HAND_CONNECTIONS:
        idx1, idx2 = connection
        if idx1 < len(landmarks) and idx2 < len(landmarks):
            lm1, lm2 = landmarks[idx1], landmarks[idx2]
            cx1, cy1 = int(lm1.x * w), int(lm1.y * h)
            cx2, cy2 = int(lm2.x * w), int(lm2.y * h)
            cv2.line(image, (cx1, cy1), (cx2, cy2), (255, 255, 255), 2)
            
    for landmark in landmarks:
        cx, cy = int(landmark.x * w), int(landmark.y * h)
        cv2.circle(image, (cx, cy), 5, (0, 0, 255), -1)

---

# 3. COUNTDOWN UTILITY

### RUN_COUNTDOWN

In [14]:
def run_countdown(cap, seconds, message):
    """Displays a countdown on the webcam feed while keeping MediaPipe active."""
    start_time = time.time()
    while time.time() - start_time < seconds:
        ret, frame = cap.read()
        if not ret: break
        
        frame = cv2.flip(frame, 1)
        
        # --- NEW: Process MediaPipe during the countdown ---
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        detection_result = detector.detect(mp_image)
        
        # Draw the skeleton if hands are visible
        if detection_result.hand_landmarks:
            for i, hand_landmarks in enumerate(detection_result.hand_landmarks):
                if i > 1: break
                draw_custom_landmarks(frame, hand_landmarks)
        # ---------------------------------------------------

        time_left = int(seconds - (time.time() - start_time)) + 1
        
        cv2.rectangle(frame, (0, 0), (640, 100), (0, 0, 0), -1)
        cv2.putText(frame, f"{message}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
        cv2.putText(frame, f"Position Hands In: {time_left}s", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
        
        cv2.imshow('Data Collection', frame)
        
        # If user presses 'q' during countdown, allow graceful exit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

---

# 4. MAIN COLLECTION LOOP

### COLLECT_DATA

In [17]:
def collect_data():
    columns = ['label', 'split'] + [f'coord_{i}' for i in range(126)]

    # 1. Load main dataset
    if os.path.exists(CSV_PATH):
        df = pd.read_csv(CSV_PATH).copy()
        print(f"Loaded existing main dataset with {len(df)} rows.")
    else:
        print("Error: Could not find existing CSV. Run extraction script first.")
        return

    # --- NEW: Load the session dataset so it persists across script restarts ---
    if os.path.exists(NEW_CSV_PATH):
        session_df = pd.read_csv(NEW_CSV_PATH).copy()
        print(f"Loaded existing session dataset with {len(session_df)} rows.")
    else:
        session_df = pd.DataFrame(columns=columns)

    cap = cv2.VideoCapture(0)
    
    while True:
        print("\n" + "="*40)
        label = input("Enter the word/sign you want to record (or 'q' to quit): ").strip().lower()
        if label == 'q':
            break
        if not label:
            continue

        new_rows = []
        splits = [('train', 40), ('val', 10)]
        is_paused = False 

        for split_name, required_frames in splits:
            captured_frames = 0
            print(f"\nRecording {required_frames} isolated frames for {split_name.upper()}...")
            
            while captured_frames < required_frames:
                msg = f"{split_name.upper()} Image {captured_frames + 1}/{required_frames}"
                run_countdown(cap, 5, msg)
                
                valid_frame_captured = False
                hand_detected_time = None 
                
                while not valid_frame_captured:
                    ret, frame = cap.read()
                    if not ret: break

                    frame = cv2.flip(frame, 1)
                    
                    key = cv2.waitKey(1) & 0xFF
                    if key == ord('q'):
                        print("Emergency exit triggered.")
                        cap.release()
                        cv2.destroyAllWindows()
                        return
                    elif key == ord('w'):
                        is_paused = not is_paused 
                        hand_detected_time = None 
                    
                    if is_paused:
                        cv2.rectangle(frame, (0, 0), (640, 100), (255, 0, 0), -1) 
                        cv2.putText(frame, "PAUSED - PRESS 'W' TO RESUME", 
                                    (60, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 3)
                        cv2.imshow('Data Collection', frame)
                        continue 

                    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
                    
                    detection_result = detector.detect(mp_image)
                    hand_features = np.zeros(126)
                    
                    if detection_result.hand_landmarks:
                        if hand_detected_time is None:
                            hand_detected_time = time.time()
                            
                        elapsed_hold_time = time.time() - hand_detected_time
                        
                        for i, hand_landmarks in enumerate(detection_result.hand_landmarks):
                            if i > 1: break
                            draw_custom_landmarks(frame, hand_landmarks)

                        if elapsed_hold_time >= 2.0:
                            for i, hand_landmarks in enumerate(detection_result.hand_landmarks):
                                if i > 1: break
                                coords = []
                                for landmark in hand_landmarks:
                                    coords.extend([landmark.x, landmark.y, landmark.z])
                                    
                                start_idx = i * 63
                                end_idx = start_idx + 63
                                hand_features[start_idx:end_idx] = coords
                            
                            row = [label, split_name] + hand_features.tolist()
                            new_rows.append(row)
                            captured_frames += 1
                            valid_frame_captured = True
                            
                            cv2.rectangle(frame, (0, 0), (640, 100), (0, 255, 0), -1)
                            cv2.putText(frame, f"IMAGE {captured_frames} CAPTURED!", 
                                        (150, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 3)
                            cv2.imshow('Data Collection', frame)
                            cv2.waitKey(1000) 
                            
                        else:
                            time_left = 2.0 - elapsed_hold_time
                            cv2.rectangle(frame, (0, 0), (640, 100), (0, 165, 255), -1) 
                            cv2.putText(frame, f"HOLD STILL: {time_left:.1f}s", 
                                        (150, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 3)
                            cv2.imshow('Data Collection', frame)
                            
                    else:
                        hand_detected_time = None
                        cv2.rectangle(frame, (0, 0), (640, 100), (0, 0, 255), -1) 
                        cv2.putText(frame, "WAITING FOR HANDS...", 
                                    (150, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 3)
                        cv2.imshow('Data Collection', frame)

        # 3. Save to datasets
        # new_df = pd.DataFrame(new_rows, columns=columns)
        # df = pd.concat([df, new_df], ignore_index=True)
        # df.to_csv(CSV_PATH, index=False)
        
        # --- NEW: Append to session dataframe and save ---
        new_df = pd.DataFrame(new_rows, columns=columns)
        session_df = pd.concat([session_df, new_df], ignore_index=True)
        session_df.to_csv(NEW_CSV_PATH, index=False)
        
        print(f"Success! Added 50 highly stable rows for '{label}'.")
        # print(f" -> Updated Main Dataset: {CSV_PATH}")
        print(f" -> Updated Session Dataset: {NEW_CSV_PATH}")

    cap.release()
    cv2.destroyAllWindows()
    print("\nData collection complete. Don't forget to run train_model.py to update your brain!")

---

In [20]:
if __name__ == "__main__":
    collect_data()

Loaded existing main dataset with 497 rows.
Loaded existing session dataset with 100 rows.


Recording 40 isolated frames for TRAIN...
Emergency exit triggered.
